# 26SU-2 Archived Low-Priority Cells

These cells were moved out of the main `26SU-2.ipynb` flow during notebook cleanup. They are retained for reference and can be copied back if needed.


## Archived Previous-Result Contour Visualization

Older local comparison plotting retained for reference.


In [ ]:
# PREVIOUS_RESULT_CONTOUR_VISUALIZATION
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import RMFsolver.constants as const

# Shared comparison controls. These default to the analytic plot settings if that cell was run first.
previous_result_temperature_axis_scale = globals().get("comparison_temperature_axis_scale", "linear")
previous_result_contour_levels_m_s = globals().get("comparison_contour_levels_m_s", [50, 100, 200, 500, 1000, 2000, 5000])
previous_result_use_fixed_level_colors = globals().get("comparison_use_fixed_level_colors", True)
previous_result_contour_colors = globals().get(
    "comparison_contour_colors",
    {
        50.0: "#1f77b4",
        100.0: "#17becf",
        200.0: "#ff7f0e",
        500.0: "#8c6d31",
        1000.0: "#2ca02c",
        2000.0: "#9467bd",
        5000.0: "#d62728",
    },
)
previous_result_contour_cmap = globals().get("comparison_contour_cmap", "rainbow")
previous_result_contour_linewidth = globals().get("comparison_contour_linewidth", 1.7)
previous_result_contour_label_fontsize = globals().get("comparison_contour_label_fontsize", 11)
previous_result_axis_fontsize = globals().get("comparison_axis_fontsize", 15)
previous_result_title_fontsize = globals().get("comparison_title_fontsize", 17)
previous_result_tick_fontsize = globals().get("comparison_tick_fontsize", 12)
previous_result_figure_size = globals().get("comparison_figure_size", (6.0, 4.2))
previous_result_x_limits = globals().get("comparison_x_limits", (0.01, 0.9))
previous_result_y_limits = globals().get("comparison_y_limits", (0.01, 100.0))
previous_result_shade_color = "grey"
previous_result_shade_alpha = 0.6
previous_result_auto_contour_count = globals().get("comparison_auto_contour_count", 7)

# Edit these if a label still lands somewhere awkward after changing levels.
previous_result_label_positions = globals().get(
    "previous_result_label_positions",
    {
        50.0: (0.08, 46.0),
        100.0: (0.14, 72.0),
        200.0: (0.30, 63.0),
        500.0: (0.43, 61.0),
        1000.0: (0.55, 56.0),
        2000.0: (0.77, 20.0),
        5000.0: (0.755, 5.0),
    },
)

n0 = 0.16 * const.MeV_fm**3
nc3 = 3.0 * n0

TT = np.logspace(np.log10(0.01), np.log10(100.0), num=160)
N = 30

V3 = np.load("/Users/janshao/NSM_related/archived_pre_2026/npy/v_n_crit=3_TTpiece__v8.npy")
Deln_cut3, T_cut3 = np.load("/Users/janshao/NSM_related/archived_pre_2026/npy/Shade_n_crit=3_TTpiece__v8.npy", allow_pickle=True)
D_dashed, T_dashed = np.load("/Users/janshao/NSM_related/archived_pre_2026/npy/D_line_n_crit=3__v2.npy")
Cut3 = interp1d(T_cut3, Deln_cut3, kind="linear", fill_value="extrapolate")

t = np.linspace(0.0, 1.0, N)
Dlo = 0.01 * n0
Dhi = 1.0 * n0
Dmax = np.minimum(Dhi, Cut3(TT) - 0.005 * n0)
Dmax = np.maximum(Dmax, Dlo)

D_grid = Dlo + (Dmax - Dlo)[:, None] * t[None, :]
T_grid = np.repeat(TT[:, None], N, axis=1)


def _previous_contour_levels(values, manual_levels=None, target_count=7):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values) & (values > 0.0)]
    if values.size < 2:
        return np.array([], dtype=float)
    vmin = float(np.nanmin(values))
    vmax = float(np.nanmax(values))
    if np.isclose(vmin, vmax):
        return np.array([], dtype=float)
    if manual_levels is not None:
        levels = np.asarray(manual_levels, dtype=float)
        levels = levels[np.isfinite(levels) & (levels > 0.0)]
        levels = np.unique(levels)
        return levels[(levels >= vmin) & (levels <= vmax)]
    if vmax / vmin >= 20.0:
        raw = np.geomspace(vmin, vmax, target_count)
    else:
        raw = np.linspace(vmin, vmax, target_count)
    return np.unique(np.asarray(raw, dtype=float))


def _previous_contour_label(value):
    value = float(value)
    if abs(value) >= 1.0e4:
        exponent = int(np.floor(np.log10(abs(value))))
        mantissa = value / (10.0 ** exponent)
        mantissa_text = f"{mantissa:.1f}".rstrip("0").rstrip(".")
        return rf"${mantissa_text}\times10^{{{exponent}}}$"
    if abs(value) >= 100.0:
        return f"{value:.0f}"
    if abs(value) >= 10.0:
        return f"{value:.1f}".rstrip("0").rstrip(".")
    return f"{value:.2g}"


def _previous_colors_for_levels(levels, color_map, use_fixed_colors=True):
    if not use_fixed_colors:
        return None
    colors = []
    for level in levels:
        matching_color = None
        for color_level, color in color_map.items():
            if np.isclose(float(level), float(color_level)):
                matching_color = color
                break
        if matching_color is None:
            return None
        colors.append(matching_color)
    return colors


def _nearest_previous_contour_point(contour, level, target_position, x_limits, y_limits):
    level_indices = np.where(np.isclose(np.asarray(contour.levels, dtype=float), float(level)))[0]
    if level_indices.size == 0:
        return target_position
    x_span = max(float(x_limits[1] - x_limits[0]), 1.0e-12)
    y_span = max(float(y_limits[1] - y_limits[0]), 1.0e-12)
    target_x, target_y = map(float, target_position)
    best_point = np.asarray([target_x, target_y], dtype=float)
    best_distance = np.inf
    for segment in contour.allsegs[int(level_indices[0])]:
        segment = np.asarray(segment, dtype=float)
        if segment.ndim != 2 or segment.shape[0] == 0:
            continue
        distances = ((segment[:, 0] - target_x) / x_span) ** 2 + ((segment[:, 1] - target_y) / y_span) ** 2
        idx = int(np.nanargmin(distances))
        if float(distances[idx]) < best_distance:
            best_distance = float(distances[idx])
            best_point = segment[idx]
    return float(best_point[0]), float(best_point[1])


previous_result_temperature_axis_scale = str(previous_result_temperature_axis_scale).lower()
if previous_result_temperature_axis_scale not in ("linear", "log"):
    raise RuntimeError("previous_result_temperature_axis_scale must be 'linear' or 'log'")
if previous_result_temperature_axis_scale == "log" and previous_result_y_limits[0] <= 0.0:
    raise RuntimeError("Log temperature axis requires a positive lower y-limit")

levels = _previous_contour_levels(
    V3,
    manual_levels=previous_result_contour_levels_m_s,
    target_count=int(previous_result_auto_contour_count),
)

fig, ax = plt.subplots(figsize=previous_result_figure_size, constrained_layout=True, facecolor="white")
ax.set_facecolor("white")
ax.grid(False)

Dcut = Cut3(TT)
ax.fill_betweenx(
    TT,
    Dcut / n0,
    previous_result_x_limits[1],
    color=previous_result_shade_color,
    alpha=float(previous_result_shade_alpha),
    zorder=0,
)

if levels.size:
    contour_colors = _previous_colors_for_levels(
        levels,
        previous_result_contour_colors,
        use_fixed_colors=bool(previous_result_use_fixed_level_colors),
    )
    contour_kwargs = {
        "levels": levels,
        "linewidths": float(previous_result_contour_linewidth),
        "zorder": 3,
    }
    if contour_colors is None:
        contour_kwargs["cmap"] = previous_result_contour_cmap
    else:
        contour_kwargs["colors"] = contour_colors
    cs = ax.contour(D_grid / n0, T_grid, V3, **contour_kwargs)
    level_color_lookup = {}
    if contour_colors is not None:
        level_color_lookup = {float(level): color for level, color in zip(cs.levels, contour_colors)}

    labeled_levels = []
    for requested_level, manual_position in previous_result_label_positions.items():
        matching_levels = [level for level in cs.levels if np.isclose(float(level), float(requested_level))]
        if not matching_levels:
            continue
        level = matching_levels[0]
        label_position = _nearest_previous_contour_point(
            cs,
            level,
            manual_position,
            previous_result_x_limits,
            previous_result_y_limits,
        )
        ax.text(
            label_position[0],
            label_position[1],
            _previous_contour_label(level),
            color=level_color_lookup.get(float(level), "0.1"),
            fontsize=int(previous_result_contour_label_fontsize),
            ha="center",
            va="center",
            zorder=5,
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.86, "pad": 0.8},
        )
        labeled_levels.append(level)

    remaining_levels = [
        level
        for level in cs.levels
        if not any(np.isclose(float(level), float(done_level)) for done_level in labeled_levels)
    ]
    if remaining_levels:
        label_texts = ax.clabel(
            cs,
            levels=remaining_levels,
            inline=True,
            inline_spacing=8,
            fontsize=int(previous_result_contour_label_fontsize),
            fmt={level: _previous_contour_label(level) for level in remaining_levels},
        )
        for text in label_texts:
            text.set_bbox({"facecolor": "white", "edgecolor": "none", "alpha": 0.86, "pad": 0.8})
else:
    ax.text(
        0.03,
        0.97,
        "no finite velocity contours in selected levels",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=int(previous_result_contour_label_fontsize),
        color="0.25",
        bbox={"facecolor": "white", "edgecolor": "0.8", "alpha": 0.9, "pad": 3.0},
    )

ax.set_xscale("linear")
ax.set_yscale(previous_result_temperature_axis_scale)
ax.set_xlim(*previous_result_x_limits)
ax.set_ylim(*previous_result_y_limits)
ax.set_xlabel(r"$(n_B^{\bar N} - n_B^{\rm crit}) / n_0$", fontsize=int(previous_result_axis_fontsize))
ax.set_ylabel(r"Temperature [MeV]", fontsize=int(previous_result_axis_fontsize))
ax.set_title(r"archived: $n_B^{\rm crit}(T=0) = 3.0\,n_0$", fontsize=int(previous_result_title_fontsize))
ax.tick_params(which="both", direction="in", top=True, right=True, length=5, width=1.0, labelsize=int(previous_result_tick_fontsize))

plt.savefig("old_contour.pdf", bbox_inches="tight", dpi=200)
plt.show()


## Archived Zero-Temperature Limit Check

Older diagnostic convergence check for the exact-zero `T_Qstar` path.


In [ ]:
# UNMAX_ZERO_TEMPERATURE_LIMIT_TEST
# Set run_uNmax_zero_limit_test=True to execute this optional direct-solver check.

run_uNmax_zero_limit_test = False
zero_limit_targets_MeV = (1.0e-1, 1.0e-2, 1.0e-3, 0.0)
zero_limit_test_rows = []

if run_uNmax_zero_limit_test:
    test_B14 = float(globals().get("B14", 180.0))
    test_nB_N = float(globals().get("nB_crit", 3.0 * n0))
    for target in zero_limit_targets_MeV:
        result = solve_front_energy_conserving_uNmax(
            T=10.0,
            nB_N=test_nB_N,
            B_one_forth=test_B14,
            TQstar=float(target),
            ms=float(ms),
            param=param,
            NM_type=NM_type,
            tail_eps=1.0e-4,
            n_mesh=30,
            tol_bvp=5.0e-2 if float(target) == 0.0 else 1.0e-2,
            max_nodes=800,
            jB_guess=0.6,
            TQ_guess=20.0,
            jB_bounds=(1.0e-4, 5.0),
            return_profile=False,
            verb=False,
        )
        zero_limit_test_rows.append({
            "TQstar_target_MeV": float(target),
            "success": bool(result.get("success", False)),
            "message": str(result.get("message", "")),
            "u_N": result.get("u_N", np.nan),
            "u_N_m_s": float(result.get("u_N", np.nan)) * const.m_per_sec,
            "jB": result.get("jB", np.nan),
            "aQstar": result.get("aQstar", np.nan),
            "T_Qstar_MeV": result.get("T_Qstar", np.nan),
            "invD_Q": result.get("invD_Q", np.nan),
        })

    zero_limit_test_df = pd.DataFrame(zero_limit_test_rows)
    display(zero_limit_test_df)
    if not zero_limit_test_df["success"].all():
        raise RuntimeError("At least one zero-limit direct target failed")
    u_values = zero_limit_test_df["u_N"].to_numpy(dtype=float)
    if not np.all(np.isfinite(u_values)) or np.any(u_values <= 0.0) or np.max(u_values) >= 1.0:
        raise RuntimeError(f"u_N did not remain finite and bounded: {u_values}")
else:
    print("Set run_uNmax_zero_limit_test=True to run the direct zero-temperature check.")
